# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities are referenced by their `@id` as per the Croissant schema for maximum transparency and reproducibility.

### Dataset Source
The dataset is described by a [Croissant schema](https://mlcommons.org/croissant/) at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load the dataset metadata and records from the Croissant schema using `mlcroissant`. All access to the dataset uses the schema URL. This automatically retrieves record set and field definitions for programmatic exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset Croissant schema.
dataset = mlc.Dataset(croissant_url)

# Show dataset title and description from the Croissant metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Croissant identifier: {metadata.identifier}")

## 2. Data Overview

Review the available record sets and their corresponding fields and columns. Extract all entities exclusively by their `@id`. This way, we can reference them in downstream extraction and processing steps.

The `RecordSet` and `Field` objects can be found in the Croissant schema, accessible via `dataset.metadata.record_sets`.

In [ ]:
# List available Record Sets and their IDs
print("Available record sets (by @id & name):")
record_set_ids = []
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    record_set_ids.append(rs.id)
    # List fields
    if rs.fields:
        print(f"  Fields (by @id & name):")
        for f in rs.fields:
            print(f"    - @id: {f.id}, name: {f.name}, dataType: {f.data_type}")
    if rs.columns:
        print(f"  Columns (by @id & name):")
        for c in rs.columns:
            print(f"    - @id: {c.id}, name: {c.name}")
    print()

## 3. Data Extraction

Load data from a record set into a DataFrame for analysis. Choose the record set `@id` you wish to work with (from the overview above). All extraction references the record set and fields by their `@id`.

Here we extract **all** available record sets for convenience.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id} ...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()[:10]}{'...' if len(df.columns)>10 else ''}")
    else:
        print(f"No records found in record set {record_set_id}.")
print("")
if dataframes:
    # Select the first tabular record set for further demonstration
    first_rs = list(dataframes.keys())[0]
    print(f"First available DataFrame is for RecordSet @id: {first_rs}")
    display(dataframes[first_rs].head())
else:
    print("No DataFrames loaded. Check dataset schema/data availability.")

## 4. Exploratory Data Analysis (EDA)

We select numeric fields for analysis, filter records according to specified criteria, normalize relevant fields, and optionally group by categorical fields. All field references are exclusively by their `@id`.

In this example, we use the first record set and detect a numeric field automatically (adjust as needed using the field list from the record set above).

In [ ]:
# Pick a DataFrame and a numeric field for analysis
record_set_for_eda = first_rs
df = dataframes[record_set_for_eda]

# Pick a numeric field by inspecting data types in the metadata
numeric_candidate_ids = []
candidate_names = []
for rs in metadata.record_sets:
    if rs.id == record_set_for_eda and rs.fields:
        for f in rs.fields:
            if f.data_type in ['schema:Integer', 'schema:Float', 'schema:Number', 'Float', 'Integer', 'Number', 'number', 'float', 'int']:
                numeric_candidate_ids.append(f.id)
                candidate_names.append(f.name)

print(f"Numeric fields (by @id): {numeric_candidate_ids}")

# Use first numeric field found
if numeric_candidate_ids:
    numeric_field_id = numeric_candidate_ids[0]
    print(f"Using numeric field: @id={numeric_field_id}, name={candidate_names[0]}")
else:
    numeric_field_id = df.columns[0]
    print(f"No explicit numeric field in metadata; defaulting to first field: {numeric_field_id}")

# Simple outlier threshold for demonstration (use 10 if plausible, else use median)
try:
    th = 10 if df[numeric_field_id].dtype.kind in ['i', 'f'] and (df[numeric_field_id].max()>10) else df[numeric_field_id].median()
except Exception:
    th = 10

# Filtering
filtered_df = df.copy()
if numeric_field_id in df.columns and df[numeric_field_id].dtype.kind in ['i', 'f']:
    filtered_df = df[df[numeric_field_id] > th]
    print(f"Filtered records with {numeric_field_id} > {th}:")
    display(filtered_df.head())

    # Normalization
    mean = filtered_df[numeric_field_id].mean()
    std = filtered_df[numeric_field_id].std()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std!=0 else 0
    print(f"Normalized {numeric_field_id} for filtered records (mean={mean:.2f}, std={std:.2f}):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Could not find numeric field {numeric_field_id} for filtering and normalization.")

# Optionally, group by a categorical field if available
group_field_id = None
all_field_ids = df.columns.tolist()
for rs in metadata.record_sets:
    if rs.id == record_set_for_eda and rs.fields:
        for f in rs.fields:
            if f.data_type in ['schema:Text', 'Text', 'string', 'categorical', 'schema:Category'] and f.id in all_field_ids:
                group_field_id = f.id
                break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped mean by categorical field {group_field_id}:")
    display(grouped_df.head())
else:
    print("No categorical group field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field for the filtered records. If there is a group field, also show mean or boxplot by group.

In [ ]:
# Visualize the filtered and normalized field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id in filtered_df.columns and filtered_df[numeric_field_id].dtype.kind in ['i','f']:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id} (> {th})")
    plt.xlabel(numeric_field_id)
    plt.show()

    # Normalized column
    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30, color='orange')
        plt.title(f"Distribution of normalized {numeric_field_id}")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()

    # Boxplot by group if available
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

We have demonstrated how to load, explore, filter, and visualize the FAIR² mass spectrometry dataset using the `mlcroissant` library. All data entities (record sets, fields, columns) were referenced by their unique `@id` per the Croissant schema specification, ensuring fully FAIR analytics.

- The Croissant schema enables dynamic, transparent access to all dataset elements via their IDs.
- Schema-driven EDA (exploratory data analysis) is reproducible and seamless using `mlcroissant` and Python libraries.
- For advanced analysis (clustering, machine learning, etc.), extend this workflow with domain-specific data science techniques referencing entities by their `@id`.

For more info, see [FAIR² project at SenScience](https://sen.science/) and the [`mlcroissant` documentation](https://mlcommons.org/croissant/).